In [1]:
import pandas as pd
import numpy as np
import cvxpy as cp
import os
import warnings
from pandas.tseries.offsets import DateOffset

warnings.filterwarnings('ignore')

def otimizador_markowitz_com_friccao(mu, cov_matrix, pesos_atuais, aversao_risco=None, custo_transacional=0.005):
    """
    Otimizador de Mínima Variância Global com Fricção (L1).
    Ignora a projeção de retornos e foca-se puramente na carteira de menor risco possível.
    """
    n = len(mu)
    
    # 1. Declaração dos Pesos
    pesos = cp.Variable(n)
    
    # 2. Equação de Risco (Ignoramos o 'mu' de Retorno Esperado)
    risco_portfolio = cp.quad_form(pesos, cov_matrix)
    
    # 3. Penalização L1 (Custos de Transação)
    giro_l1 = cp.norm(pesos - pesos_atuais, 1)
    custo_total = custo_transacional * giro_l1
    
    # 4. Função Objetivo (Mínima Variância) -> Minimizar Risco e Custos
    funcao_objetivo = cp.Minimize(risco_portfolio + custo_total)
    
    # 5. Restrições (Soma = 100%, Apenas Long)
    restricoes = [
        cp.sum(pesos) == 1,
        pesos >= 0
    ]
    
    problema = cp.Problem(funcao_objetivo, restricoes)
    try:
        problema.solve(solver=cp.OSQP)
    except:
        problema.solve(solver=cp.ECOS)
        
    return pesos.value

    
def simular_janelas_expansivas_flexivel(diretorio_dados, taxa_custo=0.005, aversao_risco=None, anos_treino=5, frequencia='Q'):
    mapa_freq_nome = {'M': 'Mensal', 'Q': 'Trimestral', 'Y': 'Anual'}
    mapa_pandas = {'M': 'ME', 'Q': 'QE', 'Y': 'YE'}
    freq_pandas = mapa_pandas.get(frequencia, frequencia)
    
    print(f"=== INÍCIO DA OTIMIZAÇÃO L1 (MÍNIMA VARIÂNCIA) | TREINO: {anos_treino} ANOS | FREQUÊNCIA: {mapa_freq_nome.get(frequencia, 'Personalizada')} ===")
    
    caminho_entrada = os.path.join(diretorio_dados, "matriz_premio_risco.csv")
    df_er = pd.read_csv(caminho_entrada, index_col='Data', parse_dates=True)
    
    colunas_ativos = [col for col in df_er.columns if col != 'Premio_Mercado_IBOV']
    df_ativos = df_er[colunas_ativos]
    n_ativos = len(colunas_ativos)
    
    datas_possiveis = df_ativos.resample(freq_pandas).last().index
    data_inicio_base = df_ativos.index[0]
    data_fim_treino = data_inicio_base + DateOffset(years=anos_treino)
    datas_rebalanceamento = datas_possiveis[datas_possiveis >= data_fim_treino]
    
    pesos_atuais = np.ones(n_ativos) / n_ativos
    df_pesos_historicos = pd.DataFrame(columns=colunas_ativos)
    
    print(f"1. A processar {len(datas_rebalanceamento)} períodos de rebalanceamento defensivo...")
    
    contador = 0
    for data_alvo in datas_rebalanceamento:
        contador += 1
        
        df_historico_conhecido = df_ativos.loc[:data_alvo]
        if len(df_historico_conhecido) < 252:
            continue
            
        mu_expansivo = df_historico_conhecido.mean() * 252
        cov_expansivo = df_historico_conhecido.cov() * 252
        
        pesos_otimizados = otimizador_markowitz_com_friccao(
            mu_expansivo, 
            cov_expansivo.values, 
            pesos_atuais, 
            custo_transacional=taxa_custo
        )
        
        df_pesos_historicos.loc[data_alvo] = pesos_otimizados
        pesos_atuais = pesos_otimizados
        
        if contador % 10 == 0 or contador == len(datas_rebalanceamento):
            print(f"   -> Progresso: {contador}/{len(datas_rebalanceamento)} otimizações concluídas ({data_alvo.date()})")
        
    # --- NOVO NOME DE FICHEIRO: MIN_VAR ---
    nome_arquivo = f"historico_pesos_l1_{anos_treino}anos_{frequencia}_min_var.parquet"
    caminho_saida = os.path.join(diretorio_dados, nome_arquivo)
    df_pesos_historicos.to_parquet(caminho_saida)
    
    print(f"\n3. Matriz de pesos defensiva gerada e salva em: {caminho_saida}")
    print("=========================================================================")
    
    return df_pesos_historicos

# ==========================================
# ÁREA DE EXECUÇÃO
# ==========================================
pasta_dados = r"C:\VSCodeWorkspace\TCC_Escrito\data"

historico_alocacao_mensal = simular_janelas_expansivas_flexivel(
    pasta_dados, 
    taxa_custo=0.005, 
    anos_treino=5, 
    frequencia='M'  
)

=== INÍCIO DA OTIMIZAÇÃO L1 (MÍNIMA VARIÂNCIA) | TREINO: 5 ANOS | FREQUÊNCIA: Mensal ===
1. A processar 132 períodos de rebalanceamento defensivo...
   -> Progresso: 10/132 otimizações concluídas (2015-10-31)
   -> Progresso: 20/132 otimizações concluídas (2016-08-31)
   -> Progresso: 30/132 otimizações concluídas (2017-06-30)
   -> Progresso: 40/132 otimizações concluídas (2018-04-30)
   -> Progresso: 50/132 otimizações concluídas (2019-02-28)
   -> Progresso: 60/132 otimizações concluídas (2019-12-31)
   -> Progresso: 70/132 otimizações concluídas (2020-10-31)
   -> Progresso: 80/132 otimizações concluídas (2021-08-31)
   -> Progresso: 90/132 otimizações concluídas (2022-06-30)
   -> Progresso: 100/132 otimizações concluídas (2023-04-30)
   -> Progresso: 110/132 otimizações concluídas (2024-02-29)
   -> Progresso: 120/132 otimizações concluídas (2024-12-31)
   -> Progresso: 130/132 otimizações concluídas (2025-10-31)
   -> Progresso: 132/132 otimizações concluídas (2025-12-31)

3. Ma